# 02 — CheXpert Label Audit

Goal: perform the dataset analysis requested in supervision on 20 October 2026.

Questions:
1. How many examples have exactly one positive observation?
2. Which conditions account for those cases?
3. What are the positive/uncertain/negative/blank counts by observation?
4. How does the dataset split across frontal and lateral views?
5. How many unique patients are represented?


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50)


## 1. Load the CheXpert CSV

In [ ]:
CSV_PATH = Path("data/CheXpert-v1.0/train.csv")
if not CSV_PATH.exists():
    raise FileNotFoundError(f"Could not find {CSV_PATH}. Update CSV_PATH to your actual CheXpert CSV.")
df = pd.read_csv(CSV_PATH)
print(f"Rows: {len(df):,}")
display(df.head())


## 2. Detect the standard observation columns

In [ ]:
OBSERVATIONS = [
    "No Finding", "Enlarged Cardiomediastinum", "Cardiomegaly", "Lung Lesion",
    "Lung Opacity", "Edema", "Consolidation", "Pneumonia", "Atelectasis",
    "Pneumothorax", "Pleural Effusion", "Pleural Other", "Fracture", "Support Devices"
]
label_cols=[c for c in OBSERVATIONS if c in df.columns]
print(f"Detected {len(label_cols)} label columns")
print(label_cols)


## 3. Audit raw label values

In [ ]:
value_audit={c: df[c].value_counts(dropna=False).sort_index() for c in label_cols}
value_audit_df=pd.DataFrame(value_audit).T
display(value_audit_df)


## 4. Label-state summary

In [ ]:
summary=pd.DataFrame(index=label_cols)
summary["positive"]=[(df[c]==1).sum() for c in label_cols]
summary["uncertain"]=[(df[c]==-1).sum() for c in label_cols]
summary["negative"]=[(df[c]==0).sum() for c in label_cols]
summary["blank"]=[df[c].isna().sum() for c in label_cols]
summary["positive_pct"]=100*summary["positive"]/len(df)
summary["uncertain_pct"]=100*summary["uncertain"]/len(df)
summary=summary.sort_values("positive",ascending=False)
display(summary.round(2))


## 5. Exactly-one-positive analysis

We compute this both across all observations and across pathology-like observations excluding `No Finding` and `Support Devices`.

In [ ]:
positive_all=(df[label_cols]==1)
n_positive_all=positive_all.sum(axis=1)
single_all=n_positive_all==1
print("ALL OBSERVATIONS")
print(f"Exactly one positive: {single_all.sum():,} ({100*single_all.mean():.2f}%)")

pathology_cols=[c for c in label_cols if c not in {"No Finding","Support Devices"}]
positive_path=(df[pathology_cols]==1)
n_positive_path=positive_path.sum(axis=1)
single_path=n_positive_path==1
print("\nPATHOLOGY-LIKE OBSERVATIONS ONLY")
print(f"Exactly one positive pathology: {single_path.sum():,} ({100*single_path.mean():.2f}%)")


## 6. Single-positive cases by condition

In [ ]:
single_counts=(positive_path.loc[single_path].sum(axis=0).sort_values(ascending=False).to_frame("single_positive_count"))
single_counts["share_pct"]=100*single_counts["single_positive_count"]/single_counts["single_positive_count"].sum()
display(single_counts.round(2))


In [ ]:
ax=single_counts["single_positive_count"].sort_values().plot(kind="barh",figsize=(8,6),title="Single-positive CheXpert examples by condition")
ax.set_xlabel("Number of examples")
ax.set_ylabel("Condition")
plt.tight_layout()
plt.show()


## 7. Frontal vs lateral composition

In [ ]:
view_col=next((c for c in ["Frontal/Lateral","View","ViewPosition"] if c in df.columns),None)
if view_col is None:
    print("No standard view column detected. Columns are:")
    print(df.columns.tolist())
else:
    print(f"Using view column: {view_col}")
    display(df[view_col].value_counts(dropna=False).to_frame("count"))


## 8. Single-positive frequency by view

In [ ]:
if view_col is not None:
    tmp=df[[view_col]].copy()
    tmp["single_positive_pathology"]=single_path
    view_summary=tmp.groupby(view_col,dropna=False)["single_positive_pathology"].agg(["count","sum","mean"])
    view_summary["single_positive_pct"]=100*view_summary["mean"]
    display(view_summary.drop(columns="mean").round(2))


## 9. Patient structure

In [ ]:
if "Path" in df.columns:
    patient=df["Path"].astype(str).str.extract(r"(patient\d+)",expand=False)
    print(f"Rows with parsed patient ID: {patient.notna().sum():,}/{len(df):,}")
    print(f"Unique parsed patients: {patient.nunique():,}")
    display(patient.value_counts().describe().to_frame("images_per_patient"))
else:
    print("No Path column detected")


## 10. Save derived summary tables

In [ ]:
OUT=Path("experiments/chexpert_audit")
OUT.mkdir(parents=True,exist_ok=True)
summary.to_csv(OUT/"label_state_summary.csv")
single_counts.to_csv(OUT/"single_positive_by_condition.csv")
if view_col is not None:
    view_summary.to_csv(OUT/"single_positive_by_view.csv")
print(f"Saved to {OUT.resolve()}")


## Interpretation prompts for Report 2

- Is the single-positive subset large enough for the first single-label experiment?
- Which conditions dominate it?
- Which conditions are too rare?
- Does excluding `No Finding` / `Support Devices` materially change the result?
- Which observations have many uncertain labels?
- How imbalanced are frontal and lateral views?
- How many images belong to the same patient?
- What does this imply for image-level vs patient-level calibration?
